# Example Spark SQL

In [1]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "Spark SQL - Example 1"

su =SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 01:26:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkUtils: Spark SQL - Example 1>

In [2]:
data = [
(1, "Alice", 29),
(2, "Bob", 35),
(3, "Charlie", 41)
]

columns = ["id", "name", "age"]

df = su._spark.createDataFrame(data, columns)

df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [3]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df_2 = su._spark.createDataFrame(data, schema)

df_2.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



# Smart Factory Sensor Data Analysis

Analysis tasks

Using SparkSQL, you need to:

Explore the schema of the DataFrame

Get the average temperature per machine

Find the maximum and minimum temperature per machine

Filter records above a temperature threshold (temp > 75)

Count the number of readings per machine

Find the machine with the highest temperature

Once you complete the Notebook with the code to solve the above tasks, generate a PDF of this Notebook and submit it to Canvas.

In [4]:
from spark_utils import SparkUtils

spark_url = "spark://spark-master:7077"
app_name = "Spark SQL - Example 1"

su =SparkUtils(spark_url, app_name)
su

<SparkUtils: Spark SQL - Example 1>

In [5]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, FloatType, TimestampType, DateType
from datetime import datetime

factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

In [6]:
factory_schema = StructType([
    StructField("id", StringType(), True),
    StructField("date", TimestampType(), True),
    StructField("temp", FloatType(), True)
])

factory_schema

StructType([StructField('id', StringType(), True), StructField('date', TimestampType(), True), StructField('temp', FloatType(), True)])

In [7]:
factory_df = su._spark.createDataFrame(factory_data, factory_schema)

factory_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- temp: float (nullable = true)



In [8]:
print(f"Number of records: {factory_df.count()}")

[Stage 0:=============================>                             (1 + 1) / 2]

Number of records: 10


In [9]:
from pyspark.sql.functions import col

df_filtered = factory_df.filter(col("temp") > 70)
df_filtered_v2 = factory_df.filter(col("temp") <= 70)

df_filtered.show(truncate = False)
df_filtered_v2.show(truncate = False)

+----+-------------------+----+
|id  |date               |temp|
+----+-------------------+----+
|M001|2026-01-26 08:00:00|75.3|
|M001|2026-01-26 08:10:00|76.1|
|M003|2026-01-26 08:15:00|72.4|
|M001|2026-01-26 08:25:00|77.5|
|M003|2026-01-26 08:30:00|73.2|
|M002|2026-01-26 08:35:00|70.1|
|M001|2026-01-26 08:40:00|78.0|
|M003|2026-01-26 08:45:00|74.6|
+----+-------------------+----+

+----+-------------------+----+
|id  |date               |temp|
+----+-------------------+----+
|M002|2026-01-26 08:05:00|68.7|
|M002|2026-01-26 08:20:00|69.8|
+----+-------------------+----+



In [10]:
factor_groupped = factory_df.groupBy(col("id")).count()
factor_groupped.show()

[Stage 7:=============================>                             (1 + 1) / 2]

+----+-----+
|  id|count|
+----+-----+
|M002|    3|
|M003|    3|
|M001|    4|
+----+-----+



In [11]:
from pyspark.sql.functions import avg, min
agg_df = factory_df.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp")
)
agg_df.show()

+----+-----------------+--------+
|  id|         avg_temp|min_temp|
+----+-----------------+--------+
|M002|69.53333282470703|    68.7|
|M003|73.39999898274739|    72.4|
|M001|76.72500038146973|    75.3|
+----+-----------------+--------+



In [12]:
from pyspark.sql.functions import avg, max, min

df_filtered_hw = factory_df.filter(col("temp") < 75)

agg_df_hw = df_filtered_hw.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    max("temp").alias("max_temp"),
    min("temp").alias("min_temp")
)
agg_df_hw.show()

factor_groupped_hw = df_filtered_hw.groupBy(col("id")).count()
factor_groupped_hw.show()

+----+-----------------+--------+--------+
|  id|         avg_temp|max_temp|min_temp|
+----+-----------------+--------+--------+
|M002|69.53333282470703|    70.1|    68.7|
|M003|73.39999898274739|    74.6|    72.4|
+----+-----------------+--------+--------+

+----+-----+
|  id|count|
+----+-----+
|M002|    3|
|M003|    3|
+----+-----+

